In [ ]:
import sys, os, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
name = torch.cuda.get_device_name(0); print("GPU:", name, flush=True)
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4, выдали {name}")
code = os.path.dirname(glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for p in glob.glob(pack + "/*"):
    dst = "/kaggle/working/pack/" + os.path.basename(p)
    if not os.path.exists(dst): os.symlink(p, dst)
hard = os.path.dirname(glob.glob("/kaggle/input/**/hard_neg_strict.parquet", recursive=True)[0])
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")

# DiTy обучена ранжировать пары на русском MS MARCO — это запросы и веб-страницы, а не
# карточки товаров. Поэтому первый этап на LLM-парах ей нужен: он объясняет, что такое
# товар. Основа та же rubert-base, что у всех наших моделей, значит на инференсе она
# попадёт в общую группу токенизатора и не будет стоить лишнего времени.
BASE = "DiTy/cross-encoder-russian-msmarco"
sys.argv = ["train_ce_large", "--prepacked", "/kaggle/working/pack", "--holdout-fold", "0",
            "--base-model", BASE,
            "--hard-negatives", f"{hard}/hard_neg_strict.parquet",
            "--epochs", "1", "--max-train-pairs", "600000",
            "--batch-size", "128", "--max-length", "256",
            "--human-epochs", "1", "--human-learning-rate", "1e-5",
            "--output", "/kaggle/working/ce_dity"]
from src.train_ce_large import main
main()
